# stareControl — リアルタイム視線推定（多点・連続学習版）

Colabの小さなセル内ではなく、**別タブの専用画面**で動作します。MediaPipe Face LandmarkerによるWebカメラ処理はブラウザ側でリアルタイムに行います。録画してから解析する方式ではありません。

## 使い方
1. ノートブックを上から順番に実行し、最後のセルのColab転送リンクから専用タブを開きます。
2. カメラを開始し、できれば**全画面**にします。
3. 離散学習は **9・25・49点**、連続学習は **30・60・90秒** から選べます。
4. **「両方学習」**を押すと、静止点の注視学習の後に連続的に動く点の追視学習が始まります。
5. 静止点は1点約1.6秒見つめます。連続追従では目標点の移動を目で追います。
6. 学習を終えると蓄積したサンプルで推定モデルを再計算し、緑の点に視線推定位置を表示します。必要に応じて**追加学習**できます。
7. 学習中の中止は **Esc** または右上の「学習を中止」。収集データは「学習サンプルCSV」から保存できます。

## 制限
通常のWebカメラによる視線座標は近似です。UIに表示する誤差は**学習データに対する誤差**であり、未知の位置への推定精度ではありません。連続学習では追視遅れを約140msと仮定してラベルを補正していますが個人差があります。推定がずれるときは姿勢・カメラ位置・照明を整えて再学習してください。

**OSカーソルの移動や実クリックはできません。** 両目を軽く細めて約0.45秒保つと画面内の疑似クリックになります。動画は録画・アップロードしません。学習データはページを閉じると消えます。MediaPipeのコードとモデルは外部CDNから取得します。

In [1]:
# 1. 独立した画面の準備
from pathlib import Path
APP_DIR = Path('/content/starecontrol-web')
APP_DIR.mkdir(parents=True, exist_ok=True)
print('作業フォルダ:', APP_DIR)

作業フォルダ: /content/starecontrol-web


## Webカメラ検出・学習・GUIの実装本体

このセルには**実装のJavaScript全文**が収録されています。9/25/49点の離散キャリブレーション、30/60/90秒の連続追従キャリブレーション、学習データの追加・初期化・CSV出力、リッジ回帰、細目クリックを含みます。

コードの原本: [web/realtime.js](https://github.com/ryouy/stareControl/blob/main/web/realtime.js)

In [2]:
%%writefile /content/starecontrol-web/realtime.js
(async function () {
  "use strict";
  if (window.__stareControlStop) { await window.__stareControlStop(); }
  const root = document.createElement("div");
  root.id = "starecontrol-realtime";
  const css = document.createElement("style");
  css.textContent = [
    "#starecontrol-realtime{position:fixed;inset:0;z-index:2147483646;background:#0a101d;color:#edf3ff;font:14px/1.5 system-ui,-apple-system,BlinkMacSystemFont,\"Segoe UI\",sans-serif;overflow:hidden;box-sizing:border-box}",
    "#starecontrol-realtime *{box-sizing:border-box}",
    "#starecontrol-realtime .panel{position:absolute;top:18px;left:18px;width:300px;max-height:calc(100% - 36px);overflow:auto;padding:22px;background:rgba(19,29,47,.96);border:1px solid #354158;border-radius:20px;box-shadow:0 15px 60px #0008;z-index:6}",
    "#starecontrol-realtime h2{margin:0;font-size:21px;letter-spacing:-.5px;font-weight:660}",
    "#starecontrol-realtime .sub{font-size:12px;color:#a5b4cd;margin:4px 0 18px}",
    "#starecontrol-realtime .section{margin:14px 0;border-top:1px solid #344059;padding-top:14px}",
    "#starecontrol-realtime button{border:1px solid #41516c;background:#24334d;color:#edf3ff;border-radius:11px;padding:10px 12px;cursor:pointer;font:inherit;font-weight:550}",
    "#starecontrol-realtime button:hover{background:#344760}",
    "#starecontrol-realtime button.primary{background:#dce9ff;border-color:#dce9ff;color:#142037}",
    "#starecontrol-realtime button:disabled{opacity:.4;cursor:not-allowed}",
    "#starecontrol-realtime .buttons{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin:8px 0}",
    "#starecontrol-realtime .buttons .wide{grid-column:span 2}",
    "#starecontrol-realtime .tiny{font-size:12px;color:#aab9d1}",
    "#starecontrol-realtime .metric{display:flex;justify-content:space-between;align-items:center;margin:7px 0;gap:12px}",
    "#starecontrol-realtime .metric strong{font-variant-numeric:tabular-nums}",
    "#starecontrol-realtime input[type=range]{width:100%;accent-color:#a8ccff;margin:7px 0 0}",
    "#starecontrol-realtime .camera{position:absolute;right:19px;top:18px;z-index:4;width:245px;background:#172438;border:1px solid #354158;border-radius:16px;padding:10px;box-shadow:0 12px 36px #0007}",
    "#starecontrol-realtime .camera-viewport{position:relative;aspect-ratio:4/3;border-radius:10px;overflow:hidden;background:#070e19}",
    "#starecontrol-realtime video,#starecontrol-realtime canvas.face{position:absolute;inset:0;width:100%;height:100%;object-fit:cover;transform:scaleX(-1)}",
    "#starecontrol-realtime video{display:block}",
    "#starecontrol-realtime .cam-caption{font-size:11px;color:#a5b6cd;margin-top:8px;text-align:center}",
    "#starecontrol-realtime .heading{position:absolute;top:40px;left:50%;transform:translateX(-50%);text-align:center;z-index:3;pointer-events:none;max-width:50vw}",
    "#starecontrol-realtime .heading h1{font-size:clamp(18px,3vw,34px);margin:0;font-weight:630;letter-spacing:-.7px}",
    "#starecontrol-realtime .heading p{color:#9eb1cc;margin:5px 0;font-size:13px}",
    "#starecontrol-realtime.calibrating .panel,#starecontrol-realtime.calibrating .camera{opacity:0;pointer-events:none}",
    "#starecontrol-realtime.calibrating .heading{top:16%}",
    "#starecontrol-realtime select{color:#eff5ff;background:#24334d;border:1px solid #465b7a;padding:7px 9px;border-radius:10px;font:inherit}",
    "#starecontrol-realtime .cancel-training{display:none;position:absolute;top:16px;right:18px;z-index:12}",
    "#starecontrol-realtime.calibrating .cancel-training{display:block}",
    "#starecontrol-realtime .target{transition:left 35ms linear,top 35ms linear}",
    "#starecontrol-realtime .target{position:absolute;width:38px;height:38px;border:2px solid #c9ddff;border-radius:50%;transform:translate(-50%,-50%);display:none;z-index:5;pointer-events:none;box-shadow:0 0 0 14px #8ebaff16,0 0 42px #96bfff55}",
    "#starecontrol-realtime .target:after{content:\"\";position:absolute;inset:12px;border-radius:100%;background:#eaf5ff}",
    "#starecontrol-realtime .cursor{position:absolute;width:26px;height:26px;border:2px solid #8ee9ce;border-radius:50%;transform:translate(-50%,-50%);display:none;z-index:4;pointer-events:none;box-shadow:0 0 0 9px #8ee9ce18,0 0 30px #8ee9ce44;transition:left 45ms linear,top 45ms linear}",
    "#starecontrol-realtime .cursor:after{content:\"\";position:absolute;inset:9px;background:#8ee9ce;border-radius:50%}",
    "#starecontrol-realtime .pulse{position:absolute;width:12px;height:12px;border:2px solid #fff;border-radius:50%;transform:translate(-50%,-50%);pointer-events:none;animation:scpulse .6s ease-out forwards;z-index:3}",
    "@keyframes scpulse{to{width:100px;height:100px;opacity:0}}",
    "#starecontrol-realtime .footer{position:absolute;bottom:14px;right:18px;max-width:min(550px,65vw);font-size:12px;color:#8194ae;text-align:right}",
    "#starecontrol-realtime .progress{height:5px;border-radius:6px;background:#2b3954;overflow:hidden;margin:8px 0}",
    "#starecontrol-realtime .progress span{display:block;height:100%;width:0%;background:#a7caff;transition:width .15s}",
    "#starecontrol-realtime .status{font-size:12px;color:#9cc4ff;min-height:36px}",
    "#starecontrol-realtime .note{font-size:11px;color:#8094b1;margin:12px 0 0}",
    "#starecontrol-realtime .pill{display:inline-block;padding:3px 8px;background:#30445b;border-radius:8px;font-size:11px;color:#bdd5f2}",
    "@media(max-width:730px){#starecontrol-realtime .panel{left:8px;top:8px;width:260px;padding:13px}#starecontrol-realtime .camera{right:8px;top:8px;width:145px}#starecontrol-realtime .heading{top:56%;max-width:70vw}#starecontrol-realtime .heading h1{font-size:18px}#starecontrol-realtime .footer{font-size:10px}}"
  ].join("");
  root.innerHTML = [
    '<div class="heading"><h1 id="sc-title">視線の推定</h1><p id="sc-instruction">カメラを開始して視線を学習してください</p></div>',
    '<div class="panel"><h2>stareControl <span class="pill">Live</span></h2><div class="sub">視線推定と細目クリック / 実験版</div>',
    '<div class="buttons"><button class="primary wide" id="sc-start">カメラを開始</button><button id="sc-cal" disabled>離散点学習</button><button id="sc-full">全画面</button><button id="sc-mode">左クリック</button><button id="sc-csv" disabled>CSV保存</button><button class="wide" id="sc-stop">終了</button></div>',
    '<div class="section"><div class="metric"><span>離散点数</span><select id="sc-grid"><option value="3">9点</option><option value="5" selected>25点</option><option value="7">49点</option></select></div><div class="metric"><span>連続学習時間</span><select id="sc-duration"><option value="30">30秒</option><option value="60" selected>60秒</option><option value="90">90秒</option></select></div><div class="buttons"><button id="sc-cont" disabled>連続追従</button><button class="primary" id="sc-both" disabled>両方学習</button><button class="wide" id="sc-clear">学習データを初期化</button><button class="wide" id="sc-traincsv" disabled>学習サンプルCSV</button></div><div class="metric"><span>学習データ</span><strong id="sc-count">0件</strong></div><div class="metric"><span>学習内の平均誤差</span><strong id="sc-fiterror">—</strong></div><p class="tiny">静止点と動く点を追って学習。学習データは追加蓄積されます。</p></div>',
    '<div class="section"><div class="metric"><span>顔の検出</span><strong id="sc-face">未開始</strong></div><div class="metric"><span>推定位置</span><strong id="sc-pos">—</strong></div><div class="metric"><span>処理速度</span><strong id="sc-fps">—</strong></div><div class="metric"><span>直近のクリック</span><strong id="sc-click">—</strong></div></div>',
    '<div class="section"><label for="sc-sensitivity">細目クリックの感度 <span id="sc-sval">78%</span></label><input id="sc-sensitivity" type="range" min="65" max="90" value="78" step="1"><p class="tiny">両目を軽く細めて約0.45秒保つとクリック。普通の瞬きは短いため無視します。</p>',
    '<label for="sc-hold">必要な維持時間 <span id="sc-hval">450 ms</span></label><input id="sc-hold" type="range" min="300" max="1000" value="450" step="50"></div>',
    '<div class="section"><div class="tiny">キャリブレーション</div><div class="progress"><span id="sc-bar"></span></div><div class="status" id="sc-status">カメラを開始してください。</div><p class="note">全画面にしてから学習してください。Escまたは右上のボタンで中止できます。推定は近似値で、OSのマウスは動きません。動画を保存・送信する処理はありません。</p></div></div>',
    '<div class="camera"><div class="camera-viewport"><video id="sc-video" autoplay playsinline muted></video><canvas class="face" id="sc-canvas" width="320" height="240"></canvas></div><div class="cam-caption">リアルタイムの目・虹彩検出</div></div>',
    '<div class="target" id="sc-target"></div><div class="cursor" id="sc-cursor"></div><button id="sc-cancel" class="cancel-training">学習を中止</button>',
    '<div class="footer">MediaPipe Face Landmarker · browser local inference · calibration required</div>'
  ].join("");
  root.appendChild(css);
  document.body.appendChild(root);
  const byId = id => root.querySelector("#" + id);
  const video=byId("sc-video"), faceCanvas=byId("sc-canvas"), faceCtx=faceCanvas.getContext("2d");
  const startBtn=byId("sc-start"), calBtn=byId("sc-cal"), fullBtn=byId("sc-full"), modeBtn=byId("sc-mode"), csvBtn=byId("sc-csv");
  const contBtn=byId("sc-cont"), bothBtn=byId("sc-both"), trainCsvBtn=byId("sc-traincsv");
  const cursor=byId("sc-cursor"), target=byId("sc-target");
  const status=msg=>{ byId("sc-status").textContent=msg; };
  let learnedSamples=[];
  const clip=x=>Math.min(1,Math.max(0,x));
  function grid(n){
    const pts=[];
    for(let y=0;y<n;y++)for(let x=0;x<n;x++)pts.push([.07+.86*x/(n-1),.07+.86*y/(n-1)]);
    const center=pts.splice(Math.floor(pts.length/2),1)[0];
    for(let i=pts.length-1;i>0;i--){const j=Math.floor(Math.random()*(i+1));[pts[i],pts[j]]=[pts[j],pts[i]];}
    return [center,...pts];
  }
  function trail(t,duration){
    const u=clip(t/duration);
    if(u<=.36){const v=u/.36;return [.5-.43*Math.cos(4*Math.PI*v),.07+.86*v];}
    if(u<=.72){const v=(u-.36)/.36;return [.07+.86*v,.5+.43*Math.cos(4*Math.PI*v)];}
    const v=(u-.72)/.28;return [.5+.43*Math.cos(4*Math.PI*v),.5+.43*Math.cos(6*Math.PI*v)];
  }
  let stream=null, detector=null, running=false, busy=false, raf=0, frames=0, countedAt=performance.now();
  let training=null, model=null, baseline=null, smoothed=null, clickMode="LEFT", squintStart=0, latched=false, lastClick=0;
  let recentOpen=[], dataset=[], lastFaceAt=0, lastFrameAt=0, lastPrediction=null, currentLandmarks=null;
  let unbindKeys=null;
  const median=arr=>{ const s=[...arr].sort((a,b)=>a-b);return s.length?s[Math.floor(s.length/2)]:NaN; };
  const mean=arr=>arr.reduce((a,b)=>a+b,0)/Math.max(arr.length,1);
  const dist=(a,b)=>Math.hypot(a.x-b.x,a.y-b.y);
  function eye(lm, spec){
    const [a,b,v1,v2,v3,v4,iris,upper,lower]=spec;
    const l=lm[a],r=lm[b], top=lm[upper], bot=lm[lower],p=lm[iris];
    const ear=(dist(lm[v1],lm[v4])+dist(lm[v2],lm[v3]))/(2*Math.max(dist(l,r),.0001));
    const xlo=Math.min(l.x,r.x), xhi=Math.max(l.x,r.x);
    const ylo=Math.min(top.y,bot.y), yhi=Math.max(top.y,bot.y);
    return {ear,gx:(p.x-xlo)/Math.max(xhi-xlo,.0001),gy:(p.y-ylo)/Math.max(yhi-ylo,.0001),ix:p.x,iy:p.y};
  }
  // 478 landmarks: iris centers 468 / 473.
  // EAR vertex ordering: outer, upper1, upper2, inner, lower2, lower1.
  function features(lm){
    if(!lm || lm.length<478)return null;
    const a=eye(lm,[33,133,160,158,153,144,468,159,145]);
    const b=eye(lm,[362,263,385,387,373,380,473,386,374]);
    const nose=lm[1],le=lm[33],re=lm[263],faceL=lm[234],faceR=lm[454],chin=lm[152],brow=lm[10];
    const width=Math.max(dist(faceL,faceR),.0001);
    const midX=(le.x+re.x)/2, midY=(le.y+re.y)/2;
    const yaw=(nose.x-midX)/Math.max(dist(le,re),.0001);
    const pitch=(nose.y-midY)/Math.max(dist(brow,chin),.0001);
    const irisX=(a.gx+b.gx)/2,irisY=(a.gy+b.gy)/2;
    return {a,b,v:[irisX,irisY,a.gx-b.gx,a.gy-b.gy,
      nose.x,nose.y,width,yaw,pitch,
      irisX*irisX,irisY*irisY,irisX*irisY]};
  }
  function solve(A,b){
    const n=b.length, M=A.map((row,i)=>[...row,b[i]]);
    for(let c=0;c<n;c++){
      let p=c;for(let r=c+1;r<n;r++)if(Math.abs(M[r][c])>Math.abs(M[p][c]))p=r;
      if(Math.abs(M[p][c])<1e-10)return null;
      [M[c],M[p]]=[M[p],M[c]];
      const pivot=M[c][c];for(let j=c;j<=n;j++)M[c][j]/=pivot;
      for(let r=0;r<n;r++)if(r!==c){const q=M[r][c];for(let j=c;j<=n;j++)M[r][j]-=q*M[c][j];}
    }
    return M.map(row=>row[n]);
  }
  function fit(samples){
    if(samples.length<35)throw new Error("有効な学習サンプルが足りません");
    const d=samples[0].v.length;
    const av=Array.from({length:d},(_,j)=>mean(samples.map(s=>s.v[j])));
    const sd=Array.from({length:d},(_,j)=>Math.max(Math.sqrt(mean(samples.map(s=>(s.v[j]-av[j])**2))),.0001));
    const rows=samples.map(s=>[1,...s.v.map((v,j)=>(v-av[j])/sd[j])]);
    const p=d+1,mat=Array.from({length:p},()=>Array(p).fill(0));
    const tx=Array(p).fill(0),ty=Array(p).fill(0);
    rows.forEach((row,i)=>{for(let j=0;j<p;j++){tx[j]+=row[j]*samples[i].x;ty[j]+=row[j]*samples[i].y;for(let k=0;k<p;k++)mat[j][k]+=row[j]*row[k];}});
    for(let j=1;j<p;j++)mat[j][j]+=6;
    mat[0][0]+=.001;
    const bx=solve(mat,tx),by=solve(mat,ty);
    if(!bx||!by)throw new Error("学習計算に失敗しました");
    return {av,sd,bx,by};
  }
  function predict(v,m=model){
    const z=[1,...v.map((x,i)=>(x-m.av[i])/m.sd[i])];
    return {x:clip(z.reduce((sum,q,i)=>sum+q*m.bx[i],0)),
            y:clip(z.reduce((sum,q,i)=>sum+q*m.by[i],0))};
  }
  function place(el,x,y){el.style.left=(x*100)+"%";el.style.top=(y*100)+"%";}
  function stageNow(){return training&&training.stages[training.stageIndex];}
  function updateProgress(now){
    if(!training)return;
    const stage=stageNow();let done=0,total=0;
    training.stages.forEach((v,i)=>{
      const d=v.kind==="grid"?v.points.length*1600:v.duration;
      total+=d;if(i<training.stageIndex)done+=d;
      else if(i===training.stageIndex)done+=v.kind==="grid"?training.point*1600+Math.min(1600,now-training.since):Math.min(d,now-training.since);
    });
    const percent=Math.round(100*done/Math.max(1,total));
    byId("sc-bar").style.width=percent+"%";
    if(stage.kind==="grid"){
      byId("sc-title").textContent="静止した点を見つめてください";
      byId("sc-instruction").textContent="点 "+(training.point+1)+" / "+stage.points.length+" · "+percent+"% · "+training.samples.length+"サンプル";
    }else{
      byId("sc-title").textContent="動く点を目で追ってください";
      byId("sc-instruction").textContent=percent+"% · "+training.samples.length+"サンプル · 顔はなるべく固定";
    }
  }
  function showStage(now){
    training.since=now;training.point=0;training.startCount=training.samples.length;
    training.lastSampleAt=0;training.retries=0;
    const stage=stageNow(),p=stage.kind==="grid"?stage.points[0]:trail(0,stage.duration);
    place(target,p[0],p[1]);target.style.display="block";updateProgress(now);
  }
  function cancelTraining(message="学習を中止しました。収集中のサンプルは追加しません。"){
    if(!training)return;
    training=null;root.classList.remove("calibrating");target.style.display="none";
    byId("sc-title").textContent="視線のリアルタイム推定";
    byId("sc-instruction").textContent="必要なら学習を再実行してください";
    byId("sc-bar").style.width="0%";status(message);
  }
  function finishTraining(){
    const fresh=training.samples;
    training=null;root.classList.remove("calibrating");target.style.display="none";
    byId("sc-title").textContent="視線のリアルタイム推定";
    byId("sc-instruction").textContent="緑の点が推定した視線位置です";
    try{
      const samples=[...learnedSamples,...fresh], next=fit(samples);
      const error=mean(samples.map(e=>{
        const p=predict(e.v,next);
        return Math.hypot((p.x-e.x)*root.clientWidth,(p.y-e.y)*root.clientHeight);
      }));
      model=next;
      // Memory cap to avoid excessive fitting cost on many extra sessions.
      learnedSamples=samples.length>7000?samples.filter((_,i)=>i%2===0):samples;
      baseline=[median(learnedSamples.map(e=>e.ea)),median(learnedSamples.map(e=>e.eb))];
      smoothed=null;latched=false;squintStart=0;lastClick=0;
      byId("sc-count").textContent=learnedSamples.length+"件";
      byId("sc-fiterror").textContent=Math.round(error)+" px (学習内)";
      byId("sc-bar").style.width="100%";trainCsvBtn.disabled=false;
      status("学習完了。新規 "+fresh.length+"件、累計 "+learnedSamples.length+"件を使いました。");
    }catch(e){byId("sc-bar").style.width="0%";status("学習エラー: "+e.message+"。以前のモデルは保持しています。");}
  }
  function nextStage(now){
    training.stageIndex++;
    if(training.stageIndex>=training.stages.length){finishTraining();return;}
    showStage(now);
  }
  function tickCalibration(info,now){
    if(!training)return;
    const stage=stageNow(),elapsed=Math.max(0,now-training.since);
    const open=info&&info.a.ear>.13&&info.b.ear>.13;
    if(stage.kind==="grid"){
      const p=stage.points[training.point];place(target,p[0],p[1]);
      if(open&&elapsed>=400&&now-training.lastSampleAt>=90){
        training.samples.push({kind:"discrete",v:info.v,x:p[0],y:p[1],ea:info.a.ear,eb:info.b.ear});
        training.lastSampleAt=now;
      }
      if(elapsed>=1600){
        if(training.samples.length-training.startCount<4){
          training.retries++;
          if(training.retries>8){cancelTraining("顔や目が十分に検出できず中断しました。");return;}
          training.since=now-700;status("顔をカメラへ向け、点を見つめてください。");return;
        }
        training.point++;
        if(training.point>=stage.points.length){nextStage(now);return;}
        training.since=now;training.startCount=training.samples.length;
        training.lastSampleAt=0;training.retries=0;
      }
    }else{
      // Approximate gaze-following latency compensation, not individually calibrated.
      const label=trail(Math.max(0,elapsed-140),stage.duration);
      place(target,...trail(elapsed,stage.duration));
      if(open&&elapsed>=500&&now-training.lastSampleAt>=90){
        training.samples.push({kind:"continuous",v:info.v,x:label[0],y:label[1],ea:info.a.ear,eb:info.b.ear});
        training.lastSampleAt=now;
      }
      if(elapsed>=stage.duration){nextStage(now);return;}
    }
    updateProgress(now);
  }
  function beginTraining(mode){
    if(!running)return;
    if(training)cancelTraining();
    const n=Number(byId("sc-grid").value),d=Number(byId("sc-duration").value)*1000,stages=[];
    if(mode==="grid"||mode==="both")stages.push({kind:"grid",points:grid(n)});
    if(mode==="continuous"||mode==="both")stages.push({kind:"continuous",duration:d});
    training={stages,stageIndex:0,since:0,point:0,startCount:0,lastSampleAt:0,retries:0,samples:[]};
    smoothed=null;squintStart=0;latched=false;
    root.classList.add("calibrating");cursor.style.display="none";
    showStage(performance.now());byId("sc-bar").style.width="0%";
  }
  function handleClick(info,now){
    if(!baseline||!model||training||!info){squintStart=0;latched=false;return;}
    const ra=info.a.ear/Math.max(baseline[0],.01), rb=info.b.ear/Math.max(baseline[1],.01);
    const limit=Number(byId("sc-sensitivity").value)/100;
    const squint=(ra+rb)/2<limit && ra<limit+.09 && rb<limit+.09;
    if(squint){
      if(!squintStart)squintStart=now;
      if(!latched && now-squintStart>=Number(byId("sc-hold").value) && now-lastClick>1100){
        latched=true;lastClick=now;
        byId("sc-click").textContent=clickMode==="LEFT"?"左クリック":"右クリック";
        status("クリック検出: "+(clickMode==="LEFT"?"左":"右")+" （ブラウザ内の疑似クリック）");
        if(smoothed){
          const pulse=document.createElement("div");pulse.className="pulse";place(pulse,smoothed.x,smoothed.y);
          root.appendChild(pulse);setTimeout(()=>pulse.remove(),700);
        }
        dataset.push({timestamp:new Date().toISOString(),event:clickMode+"_CLICK",
          x:smoothed?smoothed.x:"",y:smoothed?smoothed.y:"",earA:info.a.ear,earB:info.b.ear});
      }
    }else if((ra+rb)/2>limit+.10){squintStart=0;latched=false;}
  }
  function renderFace(lm,info){
    faceCtx.clearRect(0,0,320,240);
    if(!lm||!info)return;
    faceCtx.lineWidth=2;
    for(const id of [33,133,159,145,362,263,386,374]){
      const p=lm[id];faceCtx.fillStyle="#8bceff";faceCtx.beginPath();faceCtx.arc(p.x*320,p.y*240,2.6,0,Math.PI*2);faceCtx.fill();
    }
    for(const e of [info.a,info.b]){
      faceCtx.beginPath();faceCtx.arc(e.ix*320,e.iy*240,5,0,Math.PI*2);
      faceCtx.strokeStyle=e.ear<.18?"#ffb4a5":"#a0ffd8";faceCtx.stroke();
    }
  }
  function loop(now){
    if(!running)return;
    raf=requestAnimationFrame(loop);
    if(busy||!detector||video.readyState<2||now-lastFrameAt<33)return;
    busy=true;lastFrameAt=now;
    try{
      const r=detector.detectForVideo(video,now);
      const lm=r.faceLandmarks&&r.faceLandmarks[0]||null;
      const info=features(lm);
      renderFace(lm,info);
      if(info){
        lastFaceAt=now;byId("sc-face").textContent="検出中";
        if(!model&&!training){
          recentOpen.push([info.a.ear,info.b.ear]);if(recentOpen.length>100)recentOpen.shift();
        }
        if(training)tickCalibration(info,now);
        if(model&&!training){
          const v=predict(info.v);
          if(!smoothed)smoothed=v;
          else smoothed={x:smoothed.x*.77+v.x*.23,y:smoothed.y*.77+v.y*.23};
          place(cursor,smoothed.x,smoothed.y);cursor.style.display="block";
          lastPrediction=v;
          byId("sc-pos").textContent=Math.round(smoothed.x*100)+"%, "+Math.round(smoothed.y*100)+"%";
          handleClick(info,now);
        }else{cursor.style.display="none";}
        if(!training && model && !latched && now-lastClick>1200)status("視線を推定中（緑の点）。両目を細めると疑似クリック。");
        if(training && now-training.since<350)status(stageNow().kind==="grid"?"静止点を見つめてください":"動く点を目で追ってください");
        if(frames%4===0 && model && !training){
          dataset.push({timestamp:new Date().toISOString(),event:"GAZE",
            x:smoothed.x,y:smoothed.y,earA:info.a.ear,earB:info.b.ear});
          if(dataset.length>20000)dataset.shift();
          csvBtn.disabled=false;
        }
      }else{
        byId("sc-face").textContent="未検出";renderFace(null,null);
        if(now-lastFaceAt>250){cursor.style.display="none";squintStart=0;latched=false;}
        if(training)training.since+=33;
      }
      frames++;
      if(now-countedAt>=1000){
        byId("sc-fps").textContent=Math.round(frames*1000/(now-countedAt))+" fps";
        frames=0;countedAt=now;
      }
    }catch(e){status("処理エラー: "+e.message);console.error(e);}
    finally{busy=false;}
  }
  async function start(){
    if(running)return;
    startBtn.disabled=true;status("MediaPipeモデルを読み込んでいます…");
    try{
      const lib=await import("https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.35/+esm");
      const vision=await lib.FilesetResolver.forVisionTasks("https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.35/wasm");
      detector=await lib.FaceLandmarker.createFromOptions(vision,{
        baseOptions:{modelAssetPath:"https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",delegate:"CPU"},
        runningMode:"VIDEO",numFaces:1,outputFaceBlendshapes:false
      });
      stream=await navigator.mediaDevices.getUserMedia({video:{width:{ideal:640},height:{ideal:480},frameRate:{ideal:30}},audio:false});
      video.srcObject=stream;await video.play();
      running=true;frames=0;countedAt=performance.now();lastFrameAt=0;lastFaceAt=performance.now();
      startBtn.textContent="カメラ動作中";calBtn.disabled=false;contBtn.disabled=false;bothBtn.disabled=false;
      status("カメラを開始しました。まず「全画面」→「両方学習」をおすすめします。");
      raf=requestAnimationFrame(loop);
    }catch(e){
      status("起動できません: "+(e.message||e)+". Chromeでカメラ権限と通信状況を確認してください。");
      console.error(e);startBtn.disabled=false;startBtn.textContent="再試行";
      if(stream){stream.getTracks().forEach(t=>t.stop());stream=null;}
      if(detector){detector.close();detector=null;}
    }
  }
  async function stop(){
    running=false;cancelAnimationFrame(raf);cancelTraining();
    if(stream){stream.getTracks().forEach(t=>t.stop());stream=null;}
    if(detector){detector.close();detector=null;}
    if(document.fullscreenElement===root)await document.exitFullscreen().catch(()=>{});
    if(unbindKeys)unbindKeys();
    root.remove();window.__stareControlStop=null;
  }
  startBtn.addEventListener("click",start);
  calBtn.addEventListener("click",()=>beginTraining("grid"));
  contBtn.addEventListener("click",()=>beginTraining("continuous"));
  bothBtn.addEventListener("click",()=>beginTraining("both"));
  byId("sc-cancel").addEventListener("click",()=>cancelTraining());
  byId("sc-clear").addEventListener("click",()=>{
    cancelTraining();model=null;baseline=null;smoothed=null;learnedSamples=[];dataset=[];
    cursor.style.display="none";byId("sc-count").textContent="0件";byId("sc-fiterror").textContent="—";
    byId("sc-bar").style.width="0%";trainCsvBtn.disabled=true;csvBtn.disabled=true;
    status("蓄積学習データと推定モデルを初期化しました。");
  });
  fullBtn.addEventListener("click",async()=>{
    try{if(document.fullscreenElement===root){await document.exitFullscreen();}
        else{await root.requestFullscreen();}}
    catch(e){status("全画面にできません: "+e.message+"。画面内で学習は続行できます。");}
  });
  modeBtn.addEventListener("click",()=>{
    clickMode=clickMode==="LEFT"?"RIGHT":"LEFT";
    modeBtn.textContent=clickMode==="LEFT"?"左クリック":"右クリック";
  });
  byId("sc-sensitivity").addEventListener("input",e=>byId("sc-sval").textContent=e.target.value+"%");
  byId("sc-hold").addEventListener("input",e=>byId("sc-hval").textContent=e.target.value+" ms");
  csvBtn.addEventListener("click",()=>{
    const lines=["timestamp,event,x,y,earA,earB",...dataset.map(d=>
      [d.timestamp,d.event,d.x,d.y,d.earA,d.earB].map(v=>v==null?"":v).join(","))];
    const url=URL.createObjectURL(new Blob([lines.join("\n")],{type:"text/csv;charset=utf-8"}));
    const a=document.createElement("a");a.href=url;a.download="stareControl_realtime.csv";a.click();
    setTimeout(()=>URL.revokeObjectURL(url),2000);
  });
  trainCsvBtn.addEventListener("click",()=>{
    if(!learnedSamples.length)return;
    const cols=["kind","target_x","target_y","ear_a","ear_b",...Array.from({length:learnedSamples[0].v.length},(_,i)=>"f"+i)];
    const lines=[cols.join(","),...learnedSamples.map(v=>[v.kind,v.x,v.y,v.ea,v.eb,...v.v].join(","))];
    const url=URL.createObjectURL(new Blob([lines.join("\n")],{type:"text/csv"}));
    const link=document.createElement("a");link.href=url;link.download="starecontrol_learning_samples.csv";link.click();
    setTimeout(()=>URL.revokeObjectURL(url),2000);
  });
  byId("sc-stop").addEventListener("click",stop);
  const handleKey=e=>{
    if(e.key==="Escape"&&training){cancelTraining();return;}
    if(e.key.toLowerCase()==="r"&&!e.repeat&&!training&&document.activeElement.tagName!=="INPUT"){modeBtn.click();}
  };
  document.addEventListener("keydown",handleKey);unbindKeys=()=>document.removeEventListener("keydown",handleKey);
  window.__stareControlStop=stop;
  status("準備完了。まず「カメラを開始」を押してください。");
})().catch(err=>console.error("stareControl initialization failed",err));

Writing /content/starecontrol-web/realtime.js


## 独立画面のHTML

In [3]:
%%writefile /content/starecontrol-web/index.html
<!doctype html>
<html lang="ja">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<meta name="theme-color" content="#0a101d">
<title>stareControl | リアルタイム視線推定</title>
<style>
body{margin:0;background:#0a101d;color:#dce8fa;font:16px system-ui,-apple-system,sans-serif}
#loading{display:grid;place-items:center;min-height:100vh;text-align:center;padding:20px;box-sizing:border-box}
h1{font-weight:650;letter-spacing:-.04em}p{color:#adbfda}
</style>
<script src="./realtime.js" defer></script>
</head>
<body>
<main id="loading"><div><h1>stareControl</h1><p>画面を準備しています。スクリプトが読み込まれると操作パネルが表示されます。</p><noscript>JavaScriptを有効にしてください。</noscript></div></main>
</body>
</html>


Writing /content/starecontrol-web/index.html


## 専用画面を別タブで開く

最後のセルを実行するとColabの転送リンクが開きます。専用画面はColabの出力領域に埋め込まず、別タブで表示します。ノートブックのランタイムを停止するとこの画面も利用できなくなります。

**推奨順序:** カメラ開始 → 全画面 → 25点＋60秒の「両方学習」 → 緑の推定点を確認。

In [4]:
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from functools import partial
from threading import Thread
from google.colab import output

# 再実行時は前のWebサーバーを停止します。
if '_starecontrol_server' in globals():
    _starecontrol_server.shutdown()
    _starecontrol_server.server_close()

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass

handler = partial(QuietHandler, directory=str(APP_DIR))
_starecontrol_server = ThreadingHTTPServer(('0.0.0.0', 0), handler)
Thread(target=_starecontrol_server.serve_forever, daemon=True).start()
print('stareControl を起動しました。次のリンクから、別のブラウザタブで開いてください。')
output.serve_kernel_port_as_window(_starecontrol_server.server_port)


stareControl を起動しました。次のリンクから、別のブラウザタブで開いてください。
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>